In [31]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 6, Finished, Available, Finished, False)

In [32]:
# Read JSON Lines file from the default Lakehouse 'Files' area
# The Lakehouse name ('youtube_data') must NOT be included in the path.
# Use the relative path under the Lakehouse Files/ folder instead.

df = spark.read.json("Files/Bronze/youtube_trending_5000.jsonl")
display(df)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e92d1433-8a92-4488-b9b1-2372177368cd)

In [33]:
from pyspark.sql.functions import col, count

# Compute duplicates by video_id with record counts
videos_duplicates = (
    df.groupby("video_id")
      .agg(count("*").alias("record_count"))
      .filter(col("record_count") > 1)
)

# Optionally display the duplicates result
display(videos_duplicates)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d7d1c758-6e28-4067-82d1-b5e671961e7e)

In [34]:
repeated_records = df.join(
    videos_duplicates.select("video_id"),
    on='video_id',
    how="inner"
)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 9, Finished, Available, Finished, False)

In [35]:
display(
repeated_records.select(
    "video_id",
    "title",
    "trending_region",
    "collected_at",
    "view_count"
).orderBy("video_id", "collected_at")


)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e29c400f-2fc5-457f-a44e-4d04007237e2)

In [36]:
from pyspark.sql.functions import count, col

duplicate_observations = (
 df.groupby("video_id", "trending_region", "collected_at")
 .agg(count("*").alias("record_count"))
 .filter(col("record_count")>1)

)
print(
"Duplicate observation groups:",
duplicate_observations.count()

)

display(duplicate_observations)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 11, Finished, Available, Finished, False)

Duplicate observation groups: 0


SynapseWidget(Synapse.DataFrame, dce4f409-9344-4d56-a462-2b34d1f32ec9)

In [37]:
display(
    df.select("published_at", "collected_at").limit(5)
)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 66f059b0-4d5e-4635-85c7-b90255a7e8cb)

In [38]:
from pyspark.sql.functions import to_timestamp

# Convert string timestamps to proper timestamp type in a new DataFrame
silver_df = (
    df.withColumn("published_at", to_timestamp("published_at"))
      .withColumn("collected_at", to_timestamp("collected_at"))
)

# Inspect the updated schema to verify the types
silver_df.printSchema()

# Preview a few rows to confirm values look correct
display(
    silver_df.select(
        "video_id",
        "published_at",
        "collected_at"
    ).limit(5)
)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 13, Finished, Available, Finished, False)

root
 |-- caption: boolean (nullable = true)
 |-- category_id: string (nullable = true)
 |-- category_name: string (nullable = true)
 |-- channel_id: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- collected_at: timestamp (nullable = true)
 |-- comment_count: long (nullable = true)
 |-- definition: string (nullable = true)
 |-- description: string (nullable = true)
 |-- duration_iso: string (nullable = true)
 |-- duration_seconds: long (nullable = true)
 |-- licensed_content: boolean (nullable = true)
 |-- like_count: long (nullable = true)
 |-- published_at: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- thumbnail_url: string (nullable = true)
 |-- title: string (nullable = true)
 |-- trending_region: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- view_count: long (nullable = true)



SynapseWidget(Synapse.DataFrame, ccb7d655-3fef-4931-a7af-f6a3a5d6d15a)

In [39]:
print(
    "Invalid published at:",
    silver_df.filter(col("published_at").isNull()).count()
)

print(
    "Invalid collected at:",
    silver_df.filter(col("collected_at").isNull()).count()
)


StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 14, Finished, Available, Finished, False)

Invalid published at: 0
Invalid collected at: 0


In [40]:
from pyspark.sql.functions import trim, when, lit

silver_df = silver_df.withColumn(
"description",
when (
    col("description").isNull() |
    (trim(col("description")) ==""),
    lit("")
).otherwise(trim(col("description")))
)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 15, Finished, Available, Finished, False)

In [41]:
print(
    "Null description:",
    silver_df.filter(col("description").isNull()).count()
)

print("Silver records:", silver_df.count())

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 16, Finished, Available, Finished, False)

Null description: 0
Silver records: 2500


In [42]:
numeric_columns =[
    "view_count",
    "like_count",
    "comment_count",
    "duration_seconds"
]

for column_name in numeric_columns:
    invalid_count = silver_df.filter(
        col(column_name) < 0
    ).count()

    print(column_name, ":", invalid_count)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 17, Finished, Available, Finished, False)

view_count : 0
like_count : 0
comment_count : 0
duration_seconds : 0


In [43]:
from pyspark.sql.functions import current_timestamp

silver_df = silver_df.withColumn("processed_at", current_timestamp()
)

display(
silver_df.select(
        "video_id",
        "collected_at",
        "processed_at"
    ).limit(5)
)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7cccc4df-0b2b-4979-a006-b161b4d3de3f)

In [48]:
silver_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("silver_youtube_videos")

print("Siler Delta table created successfully")

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 23, Finished, Available, Finished, False)

Siler Delta table created successfully


In [49]:
silver_df.createOrReplaceTempView("silver_youtube_ready")

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 24, Finished, Available, Finished, False)

In [51]:
saved_silver = spark.read.table("silver_youtube_videos")

print("Silver records:", saved_silver.count())
print("Silver columns:", len(saved_silver.columns))

display(saved_silver.limit(5))

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 26, Finished, Available, Finished, False)

Silver records: 2500
Silver columns: 21


SynapseWidget(Synapse.DataFrame, 82c5e25f-60a8-49c2-9af9-b3b3eca06642)

In [53]:
from pyspark.sql.functions import sum, countDistinct

gold_category = (
    saved_silver
    .groupby("category_name")
    .agg(
        countDistinct("video_id").alias("total_videos"),
        sum("view_count").alias("total_views")
            )
    .orderBy(col("total_views").desc())
)

display(gold_category)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 21258240-08d9-49e2-84cb-ec3d06bb9425)

In [55]:
from pyspark.sql.functions import max, count, sum, col

# First, get one row per video/category with the maximum view_count
unique_videos = (
    saved_silver
    .groupBy("video_id", "category_name")
    .agg(max("view_count").alias("view_count"))
)

# Then, calculate category-level metrics from the deduplicated videos
gold_category = (
    unique_videos
    .groupBy("category_name")
    .agg(
        count("video_id").alias("total_videos"),
        sum("view_count").alias("total_views")
    )
    .orderBy(col("total_views").desc())
)

display(gold_category)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fdbd3b84-b6c2-43e4-a87e-9a78a6328ef9)

In [57]:
# Save gold_category as a managed Delta table in the default Lakehouse
# Use a string table name instead of an undefined Python variable

gold_category.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_youtube_category")

print("Gold category table successfully created")

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 32, Finished, Available, Finished, False)

Gold category table successfully created


In [58]:
saved_gold = spark.read.table("gold_youtube_category")

print("Total categories:", saved_gold.count())

display(
    saved_gold.orderBy(col("total_views").desc())
)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 33, Finished, Available, Finished, False)

Total categories: 12


SynapseWidget(Synapse.DataFrame, 1a3ef9ea-c6a0-4a3e-824d-48d7004b4acc)

In [59]:

from pyspark.sql.functions import row_number, desc, round
from pyspark.sql.window import Window

# Select the latest observation for each video
video_window = Window.partitionBy("video_id").orderBy(
    desc("collected_at"),
    desc("view_count")
)

latest_videos = (
    saved_silver
    .withColumn("row_num", row_number().over(video_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

# Calculate engagement rate
gold_engagement = latest_videos.withColumn(
    "engagement_rate",
    when(
        col("view_count") > 0,
        round(
            (col("like_count") + col("comment_count"))
            / col("view_count") * 100,
            2
        )
    )
)

display(
    gold_engagement.select(
        "video_id",
        "title",
        "category_name",
        "view_count",
        "like_count",
        "comment_count",
        "engagement_rate"
    ).orderBy(col("engagement_rate").desc()).limit(10)
)


StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8402bdd1-1bd5-41c3-aaa8-6369d5f04afb)

In [60]:
print("Unique videos:", gold_engagement.count())

print(
    "Null engagement rates:",
    gold_engagement.filter(
        col("engagement_rate").isNull()
    ).count()
)

display(
    gold_engagement.select(
        "engagement_rate"
    ).summary()
)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 35, Finished, Available, Finished, False)

Unique videos: 1952
Null engagement rates: 0


SynapseWidget(Synapse.DataFrame, 3c79b03d-d5c8-4d91-871e-94b1b430e6ea)

In [61]:

gold_engagement.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_youtube_engagement")

print("Gold engagement table saved successfully!")


StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 36, Finished, Available, Finished, False)

Gold engagement table saved successfully!


In [62]:
saved_engagement = spark.read.table("gold_youtube_engagement")

print("Saved records:", saved_engagement.count())

display(
    saved_engagement.select(
        "title",
        "category_name",
        "view_count",
        "engagement_rate"
    ).orderBy(col("engagement_rate").desc()).limit(10)
)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 37, Finished, Available, Finished, False)

Saved records: 1952


SynapseWidget(Synapse.DataFrame, 554269be-cb8b-464e-8d60-3bd7530a7d46)

In [63]:
print(
    "Videos with 0% engagement:",
    gold_engagement.filter(
        col("engagement_rate") == 0
    ).count()
)

print(
    "Videos with positive engagement:",
    gold_engagement.filter(
        col("engagement_rate") > 0
    ).count()
)

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 38, Finished, Available, Finished, False)

Videos with 0% engagement: 3
Videos with positive engagement: 1949


In [64]:
gold_engagement.filter(
    col("engagement_rate").isNull()
).count()

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 39, Finished, Available, Finished, False)

0

In [65]:
gold_engagement.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_youtube_engagement")

print("Gold engagement table saved successfully!")

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 40, Finished, Available, Finished, False)

Gold engagement table saved successfully!


In [66]:
saved_engagement = spark.read.table("gold_youtube_engagement")

print("Total records:", saved_engagement.count())

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 41, Finished, Available, Finished, False)

Total records: 1952


In [67]:

from pyspark.sql.functions import avg, max, countDistinct, round

gold_region = (
    saved_silver
    .groupBy("trending_region")
    .agg(
        countDistinct("video_id").alias("total_videos"),
        round(avg("view_count"), 2).alias("avg_views"),
        max("view_count").alias("max_views")
    )
    .orderBy(col("total_videos").desc())
)

display(gold_region)


StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8d48fa50-21dd-490a-b84b-ba21e4002afe)

In [68]:

region_window = Window.partitionBy(
    "video_id",
    "trending_region"
).orderBy(
    desc("collected_at"),
    desc("view_count")
)

latest_region_videos = (
    saved_silver
    .withColumn("row_num", row_number().over(region_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

gold_region = (
    latest_region_videos
    .groupBy("trending_region")
    .agg(
        countDistinct("video_id").alias("total_videos"),
        round(avg("view_count"), 2).alias("avg_views"),
        max("view_count").alias("max_views")
    )
    .orderBy(col("avg_views").desc())
)

display(gold_region)


StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f066ec9e-ae74-485a-b86d-dd43280056d4)

In [69]:
gold_region.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_youtube_region")

print("Regional Gold table saved successfully!")

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 44, Finished, Available, Finished, False)

Regional Gold table saved successfully!


In [70]:
saved_region = spark.read.table("gold_youtube_region")

print("Total regions:", saved_region.count())

display(saved_region.orderBy(col("avg_views").desc()))

StatementMeta(, 68e96c13-4488-4969-884f-6f194a546e20, 45, Finished, Available, Finished, False)

Total regions: 13


SynapseWidget(Synapse.DataFrame, 0d18f468-c2b6-4ae9-a708-2cf6f8ffaa9c)